# Step 1. Convert Raw EDF Data to HDF5

Convert EDF projection files acquired at the **ID16A nano-imaging beamline (ESRF)** into a single
consolidated HDF5 archive following the [Data Exchange](http://dxchange.readthedocs.io/) layout.

The beamline uses a **Fresnel zone-plate** setup: the zone plate focuses the beam onto the sample,
and the detector is placed downstream at distance `focustodetectordistance`.  
Multiple sample-to-detector distances `z1` are recorded in a single scan to provide holographic
diversity (multi-distance phase retrieval).

**Outputs written to the HDF5 file**
| Dataset | Description |
|---------|-------------|
| `/exchange/data{k}` | Raw projections at distance index `k` |
| `/exchange/data_white_start{k}` / `data_white_end{k}` | Flat fields before / after scan |
| `/exchange/data_dark{k}` | Dark fields |
| `/exchange/theta` | Rotation angles (degrees) extracted from EDF headers |
| `/exchange/shifts` | Pre-measured sample motion shifts from `correct.txt` |
| `/exchange/voxelsize` | Effective voxel sizes per distance |
| `/exchange/z1` | Sample-to-focus distances (m) |

In [ ]:
import h5py 
import fabio
import glob
import json
import os
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from holotomocupy.utils import *


def _read_h5_field(h5path, suffix):
    """Return the value of the first dataset whose path ends with `suffix`."""
    result = {}
    def _visit(name, obj):
        if not result and isinstance(obj, h5py.Dataset) and name.endswith(suffix):
            result['val'] = obj[()]
    with h5py.File(h5path, 'r') as f:
        f.visititems(_visit)
    if not result:
        raise KeyError(f'{suffix!r} not found in {h5path}')
    return result['val']


def read_energy(h5path):
    """Read X-ray energy from HDF5 TOMO metadata. Returns value in keV."""
    return float(_read_h5_field(h5path, 'TOMO/energy'))


def read_sx0(h5path):
    """Read focal-spot position sx0 from HDF5 TOMO metadata. Returns value in metres."""
    return float(_read_h5_field(h5path, 'TOMO/sx0')) * 1e-3


def read_sx(h5path):
    """Read sample sx motor position from HDF5 positioners. Returns value in metres."""
    names  = _read_h5_field(h5path, 'sample/positioners/name').decode().split()
    values = _read_h5_field(h5path, 'sample/positioners/value').decode().split()
    if 'sx' not in names:
        raise ValueError(f"'sx' not found in positioners for {h5path}.\n"
                         f"Available names: {names}")
    return float(values[names.index('sx')]) * 1e-3


def read_detector_pixelsize(h5path):
    """Read image pixel size from FTOMO_PAR JSON. Returns value in metres."""
    par = json.loads(_read_h5_field(h5path, 'TOMO/FTOMO_PAR').decode())
    return float(par['image_pixel_size']) * 1e-6


def read_focustodetectordistance(h5path):
    """Read focus-to-detector distance from PTYCHO parameters. Returns value in metres."""
    return float(_read_h5_field(h5path, 'PTYCHO/focusToDetectorDistance')) * 1e-3


def find_angle(fname):
    """Extract rotation angle from EDF header (stops at motor_pos line)."""
    with open(fname, 'rb') as f:
        buf = b''
        while b'motor_pos' not in buf:
            chunk = f.read(512)
            if not chunk:
                break
            buf += chunk
    for line in buf.decode('latin-1').split('\n'):
        if 'motor_pos' in line:
            return float(line.split()[3])

## Parameters

Edit the paths and geometry below to match your dataset.

- **`z1_ids`** – indices (0-based) of the distances to use from the full 8-distance scan.  
  Using 4 distances (`[0,1,2,3]`) gives a good balance of phase diversity and memory.
- **`sx0`** – longitudinal position of the focal spot (m), used to shift raw `z1` values to actual
  sample-to-focus distances.
- **`detector_pixelsize`** – physical pixel pitch on the detector (m); multiplied by 2 because the
  detector is operated in 2×2 binning.
- **Derived quantities** (computed automatically):
  - `z2 = focustodetectordistance - z1` — sample to detector distance  
  - `magnifications = focustodetectordistance / z1` — geometric magnification per distance  
  - `distances = z1*z2/focustodetectordistance` — effective propagation distance for phase retrieval  
  - `voxelsize = detector_pixelsize / magnification` — object-space voxel size

In [ ]:
path = '/data2/vnikitin/atomium/20240924/AtomiumS2/'
pfile = 'AtomiumS2_HT_007nm'
path_out = path.rstrip('/') + '_rec'
file_out = f'{pfile}.h5'

# Auto-detect ntheta from ref filenames: ref0000_0000.edf and ref0000_{ntheta}.edf
dname0 = f'{path}/{pfile}_1_'
ntheta = max(int(f.split('_')[-1].split('.')[0])
             for f in glob.glob(f'{dname0}/ref0000_*.edf'))

# Auto-extract geometry — one H5 file per distance directory (_1_, _2_, ...),
# excluding special scans (NFP, etc.)
dirs    = sorted(glob.glob(f'{path}/{pfile}_[0-9]_/'))
h5files = [sorted(glob.glob(f'{d}/*.h5'))[0] for d in dirs]
ndist   = len(h5files)

energy                  = read_energy(h5files[0])                           # keV
detector_pixelsize      = read_detector_pixelsize(h5files[0])               # m
focustodetectordistance = read_focustodetectordistance(h5files[0])          # m
sx0                     = read_sx0(h5files[0])                              # m, same for all distances
z1                      = np.array([read_sx(f) for f in h5files]) - sx0    # m

wavelength = 1.24e-09 / energy

# n from actual EDF file size (images are n×n)
n0, n1 = fabio.open(f'{dname0}/ref0000_0000.edf').data.shape
n = n0

# Auto-detect number of flat / dark frames by counting files
nref  = len(glob.glob(f'{dname0}/ref*_0000.*'))
ndark = len(glob.glob(f'{dname0}/darkend*.*'))

print(f'ntheta                  = {ntheta}')
print(f'energy                  = {energy} keV')
print(f'detector_pixelsize      = {detector_pixelsize} m')
print(f'focustodetectordistance = {focustodetectordistance} m')
print(f'sx0                     = {sx0} m')
print(f'z1                      = {z1} m')
print(f'ndist                   = {ndist}, n = {n}')
print(f'nref                    = {nref}, ndark = {ndark}')

z2 = focustodetectordistance - z1

distances = (z1 * z2) / focustodetectordistance
magnifications = focustodetectordistance / z1
norm_magnifications = magnifications / magnifications[0]
voxelsizes = np.abs(detector_pixelsize / magnifications)
voxelsize = voxelsizes[0]

### Parse files and save everything to h5

Iterate over all projection angles and distances.  For each distance `k` the EDF files are read,
cropped to an `n × n` region centred on the detector, and written directly into the preallocated
HDF5 datasets.  Rotation angles are extracted from the `motor_pos` field in the EDF header.

> **Tip:** Run this step once per dataset; subsequent steps read only from the HDF5 file.

In [ ]:
os.makedirs(path_out, exist_ok=True)

sty, endy = 0, n
stx, endx = 0, n

with h5py.File(f'{path_out}/{file_out}', 'w') as fid:
    data_ds   = [fid.create_dataset(f'/exchange/data{k}',             shape=(ntheta, n, n), dtype='uint16') for k in range(ndist)]
    white0_ds = [fid.create_dataset(f'/exchange/data_white_start{k}', shape=(nref,  n, n),  dtype='uint16') for k in range(ndist)]
    white1_ds = [fid.create_dataset(f'/exchange/data_white_end{k}',   shape=(nref,  n, n),  dtype='uint16') for k in range(ndist)]
    dark_ds   = [fid.create_dataset(f'/exchange/data_dark{k}',        shape=(ndark, n, n),  dtype='uint16') for k in range(ndist)]

    theta_ds  = fid.create_dataset('/exchange/theta',  shape=(ntheta, ndist), dtype='float32')
    shifts_ds = fid.create_dataset('/exchange/shifts', shape=(ntheta, ndist, 2), dtype='float32')
    attrs_ds  = fid.create_dataset('/exchange/attrs',  shape=(ntheta, ndist, 3), dtype='float32')

    fid.create_dataset('/exchange/voxelsize',             data=voxelsizes,                dtype='float32')
    fid.create_dataset('/exchange/z1',                    data=z1,                        dtype='float32')
    fid.create_dataset('/exchange/detector_pixelsize',    data=[detector_pixelsize],      dtype='float32')
    fid.create_dataset('/exchange/energy',                data=[energy],                  dtype='float32')
    fid.create_dataset('/exchange/focusdetectordistance', data=[focustodetectordistance], dtype='float32')

    # --- Rotation angles: read all EDF headers in parallel (I/O bound) ---
    fnames = [f'{dname0}/{pfile}_1_{id:04}.edf' for id in range(ntheta)]
    with ThreadPoolExecutor() as pool:
        theta_vals = np.array(list(pool.map(find_angle, fnames)), dtype='float32')
    theta_ds[:] = theta_vals[:, None]   # broadcast to all ndist columns

    for k in range(ndist):
        dname = f'{path}/{pfile}_{k + 1}_'
        shifts_ds[:, k] = np.loadtxt(f'{dname}/correct.txt')[:ntheta]
        attrs_ds[:,  k] = np.loadtxt(f'{dname}/attributes.txt')[:ntheta, :3]

        for id in range(nref):
            white0_ds[k][id] = fabio.open(f'{dname}/ref{id:04}_0000.edf').data[sty:endy, stx:endx]
            white1_ds[k][id] = fabio.open(f'{dname}/ref{id:04}_{ntheta:04}.edf').data[sty:endy, stx:endx]
        for id in range(ndark):
            dark_ds[k][id]   = fabio.open(f'{dname}/darkend{id:04}.edf').data[sty:endy, stx:endx]

        for id in range(ntheta):
            fname = f'{dname}/{pfile}_{k + 1}_{id:04}.edf'
            data_ds[k][id] = fabio.open(fname).data[sty:endy, stx:endx]
            if id % 100 == 0:
                print(id, k, theta_vals[id], shifts_ds[id, k])